# Coral Video Inference V1

## YOLO11n Stage 3 → +10% Context Crop → MobileNetV4 V3

This notebook takes the shared `coral_video.zip`, extracts all supported videos, processes them frame-by-frame, and writes annotated videos.

### Pipeline

`Video → Frame → YOLO Stage3 → Coral BBox → +10% Context → 288×288 MobileNetV4 V3 → Species → Draw → Output MP4`

### Default model configuration

- Detector: **YOLO11n Stage 3 `best.pt`**
- Detector image size: **960**
- Detector confidence: **0.35**
- Classifier: **MobileNetV4 Conv Small 0.5 — V3**
- Classifier crop: **predicted box + 10% context**
- Classifier input: **288 × 288**
- Detection class remains one class: `target_coral`
- Species is predicted by the classifier.

### Memory design

The notebook is **streaming**:
- one video at a time
- one frame at a time
- no entire-video arrays in RAM
- classifier batches only detections from the current frame

### Output

For every input video:
- annotated MP4
- frame/object CSV
- per-video summary

A final experiment summary CSV/JSON is also saved.

No training is performed.


In [ ]:
# ============================================================
# CELL 1 — INSTALL + MOUNT + IMPORTS
# ============================================================

!pip -q install ultralytics==8.4.117 timm==1.0.22

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import defaultdict

import os
import gc
import csv
import json
import math
import time
import shutil
import zipfile
import random

import numpy as np
import pandas as pd
import cv2

from PIL import Image

import torch
import timm

from timm.data import (
    resolve_model_data_config,
    create_transform,
)

from ultralytics import YOLO


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a GPU runtime in Colab before continuing."
    )

DEVICE = torch.device("cuda")
YOLO_DEVICE = 0

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print("=" * 110)
print("ENVIRONMENT")
print("=" * 110)
print("GPU  :", GPU_NAME)
print(f"VRAM : {VRAM_GB:.2f} GB")
print("Torch:", torch.__version__)
print("timm :", timm.__version__)
print("STATUS: ENVIRONMENT_READY")


In [ ]:
# ============================================================
# CELL 2 — PROJECT PATHS + FIND coral_video.zip AUTOMATICALLY
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Coral_Reef_AI_Project"
)

# ------------------------------------------------------------
# Stage3 detector
# ------------------------------------------------------------

DETECTOR_CKPT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "YOLO11n_Stage3_TargetedSliced_SmallCrowded"
    / "Detection_Stage3"
    / "weights"
    / "best.pt"
)

# ------------------------------------------------------------
# Final MobileNetV4 V3 classifier
# ------------------------------------------------------------

CLASSIFIER_CKPT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "MobileNetV4_Classification_V3_288_Context_HardMining"
    / "best_model_v3_macro_f1.pth"
)

# ------------------------------------------------------------
# Search for the video ZIP in mounted MyDrive.
#
# Preferred expected path from your Drive screenshot:
# /content/drive/MyDrive/Coral_video/coral_video.zip
# ------------------------------------------------------------

EXPECTED_ZIP = Path(
    "/content/drive/MyDrive/Coral_video/coral_video.zip"
)

MYDRIVE = Path(
    "/content/drive/MyDrive"
)


def find_coral_video_zip():
    if EXPECTED_ZIP.exists():
        return EXPECTED_ZIP

    exact_matches = list(
        MYDRIVE.rglob("coral_video.zip")
    )

    if exact_matches:
        return exact_matches[0]

    # Case-insensitive / slightly different naming fallback.
    zip_matches = [
        p
        for p in MYDRIVE.rglob("*.zip")
        if (
            "coral" in p.name.lower()
            and "video" in p.name.lower()
        )
    ]

    if zip_matches:
        return zip_matches[0]

    raise FileNotFoundError(
        "Could not find coral_video.zip in MyDrive.\n"
        "Expected something like:\n"
        "/content/drive/MyDrive/Coral_video/coral_video.zip\n\n"
        "If the folder is under 'Shared with me', first add a shortcut "
        "to My Drive or copy the ZIP into My Drive."
    )


VIDEO_ZIP = find_coral_video_zip()

for p in [
    DETECTOR_CKPT,
    CLASSIFIER_CKPT,
    VIDEO_ZIP,
]:
    if not p.exists():
        raise FileNotFoundError(p)


print("=" * 110)
print("PATH CHECK")
print("=" * 110)

print("Detector:")
print(DETECTOR_CKPT)

print("\nClassifier:")
print(CLASSIFIER_CKPT)

print("\nVideo ZIP:")
print(VIDEO_ZIP)

print("\nSTATUS: PATHS_READY")


In [ ]:
# ============================================================
# CELL 3 — EXTRACT ZIP + DISCOVER VIDEOS
# ============================================================

WORK_ROOT = Path(
    "/content/coral_video_inference"
)

INPUT_ROOT = (
    WORK_ROOT
    / "input_videos"
)

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

INPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print("Extracting:")
print(VIDEO_ZIP)

with zipfile.ZipFile(
    VIDEO_ZIP,
    "r"
) as zf:
    zf.extractall(
        INPUT_ROOT
    )


VIDEO_EXTENSIONS = {
    ".mp4",
    ".mov",
    ".avi",
    ".mkv",
    ".m4v",
    ".webm",
    ".mpg",
    ".mpeg",
}


video_paths = sorted([
    p
    for p in INPUT_ROOT.rglob("*")
    if (
        p.is_file()
        and p.suffix.lower() in VIDEO_EXTENSIONS
    )
])


if not video_paths:
    raise RuntimeError(
        "ZIP extracted successfully, but no supported video files were found."
    )


print("=" * 110)
print("VIDEOS FOUND")
print("=" * 110)

for i, p in enumerate(
    video_paths,
    start=1
):
    size_mb = (
        p.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{i:02d}. {p.relative_to(INPUT_ROOT)}"
        f"  ({size_mb:.1f} MB)"
    )


print("\nTotal videos:", len(video_paths))
print("STATUS: VIDEOS_DISCOVERED")


In [ ]:
# ============================================================
# CELL 4 — LOAD DETECTOR + CLASSIFIER
# ============================================================

DETECTOR_IMGSZ = 960
DETECTOR_CONF = 0.35
DETECTOR_NMS_IOU = 0.70

CLASSIFIER_INPUT = 288
CONTEXT_PAD = 0.10

# Classification display threshold:
# We still display the top class below this threshold,
# but mark it as LOWCONF in the label.
CLASSIFIER_LOW_CONF = 0.50


# ------------------------------------------------------------
# Detector
# ------------------------------------------------------------

detector = YOLO(
    str(DETECTOR_CKPT)
)


# ------------------------------------------------------------
# Classifier checkpoint
# ------------------------------------------------------------

cls_ckpt = torch.load(
    CLASSIFIER_CKPT,
    map_location=DEVICE,
    weights_only=False,
)

FALLBACK_MODEL_NAME = (
    "mobilenetv4_conv_small_050.e3000_r224_in1k"
)

MODEL_NAME = cls_ckpt.get(
    "model_name",
    FALLBACK_MODEL_NAME,
)


# Stable project operational labels.
DEFAULT_CLASSES = [
    "Euphflfiaancora",
    "Favosites",
    "Platygyra",
    "Sarcophyton",
    "Sinularia",
    "WavingHand",
]


class_to_idx = cls_ckpt.get(
    "class_to_idx",
    None,
)

if class_to_idx is None:
    class_to_idx = {
        c: i
        for i, c in enumerate(DEFAULT_CLASSES)
    }
else:
    class_to_idx = {
        str(k): int(v)
        for k, v in class_to_idx.items()
    }


idx_to_class = {
    int(v): str(k)
    for k, v in class_to_idx.items()
}

NUM_CLASSES = len(
    idx_to_class
)

if NUM_CLASSES != 6:
    raise RuntimeError(
        f"Expected 6 classifier classes, got {NUM_CLASSES}: "
        f"{class_to_idx}"
    )


classifier = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=NUM_CLASSES,
).to(DEVICE)

classifier.load_state_dict(
    cls_ckpt["model_state_dict"]
)

classifier.eval()


data_cfg = resolve_model_data_config(
    classifier
)

data_cfg["input_size"] = (
    3,
    CLASSIFIER_INPUT,
    CLASSIFIER_INPUT,
)

if "mean" in cls_ckpt:
    data_cfg["mean"] = tuple(
        cls_ckpt["mean"]
    )

if "std" in cls_ckpt:
    data_cfg["std"] = tuple(
        cls_ckpt["std"]
    )


cls_transform = create_transform(
    **data_cfg,
    is_training=False,
)


try:
    detector_strides = [
        int(round(float(x)))
        for x in (
            detector.model.stride
            .detach()
            .cpu()
            .tolist()
        )
    ]
except Exception:
    detector_strides = None


print("=" * 110)
print("MODELS LOADED")
print("=" * 110)

print("Detector strides :", detector_strides)
print("Detector imgsz   :", DETECTOR_IMGSZ)
print("Detector conf    :", DETECTOR_CONF)

print("\nClassifier model :", MODEL_NAME)
print("Classifier input :", CLASSIFIER_INPUT)
print("Context padding  :", CONTEXT_PAD)
print("Classes          :", class_to_idx)

print("\nSTATUS: MODELS_LOADED")


In [ ]:
# ============================================================
# CELL 5 — VIDEO INFERENCE HELPERS
# ============================================================

def padded_crop_from_xyxy(
    frame_bgr,
    box,
    padding=0.10,
):
    H, W = frame_bgr.shape[:2]

    x1, y1, x2, y2 = map(
        float,
        box
    )

    bw = max(
        0.0,
        x2 - x1
    )

    bh = max(
        0.0,
        y2 - y1
    )

    if bw <= 1 or bh <= 1:
        return None, None

    pad_x = bw * padding
    pad_y = bh * padding

    px1 = max(
        0,
        int(math.floor(x1 - pad_x))
    )

    py1 = max(
        0,
        int(math.floor(y1 - pad_y))
    )

    px2 = min(
        W,
        int(math.ceil(x2 + pad_x))
    )

    py2 = min(
        H,
        int(math.ceil(y2 + pad_y))
    )

    if px2 <= px1 or py2 <= py1:
        return None, None

    crop_bgr = frame_bgr[
        py1:py2,
        px1:px2,
    ]

    if crop_bgr.size == 0:
        return None, None

    return (
        crop_bgr,
        (
            px1,
            py1,
            px2,
            py2,
        ),
    )


def classify_frame_crops(
    crops_bgr,
):
    if not crops_bgr:
        return []

    tensors = []

    for crop_bgr in crops_bgr:
        crop_rgb = cv2.cvtColor(
            crop_bgr,
            cv2.COLOR_BGR2RGB,
        )

        pil_crop = Image.fromarray(
            crop_rgb
        )

        tensors.append(
            cls_transform(
                pil_crop
            )
        )

    batch = torch.stack(
        tensors
    ).to(
        DEVICE,
        non_blocking=True,
    )

    with torch.no_grad():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = classifier(
                batch
            )

        probs = torch.softmax(
            logits.float(),
            dim=1,
        )

        confs, pred_idx = probs.max(
            dim=1
        )

    pred_idx = pred_idx.cpu().tolist()
    confs = confs.cpu().tolist()
    probs_np = (
        probs
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    outputs = []

    for i in range(
        len(pred_idx)
    ):
        idx = int(
            pred_idx[i]
        )

        outputs.append({
            "class_index": idx,
            "species": idx_to_class[idx],
            "class_conf": float(
                confs[i]
            ),
            "probs": probs_np[i],
        })

    del batch
    del logits
    del probs

    return outputs


def draw_detection_label(
    frame,
    box,
    species,
    det_conf,
    cls_conf,
):
    H, W = frame.shape[:2]

    x1, y1, x2, y2 = [
        int(round(v))
        for v in box
    ]

    x1 = max(
        0,
        min(x1, W - 1)
    )

    y1 = max(
        0,
        min(y1, H - 1)
    )

    x2 = max(
        0,
        min(x2, W - 1)
    )

    y2 = max(
        0,
        min(y2, H - 1)
    )

    # Confidence state changes line thickness only.
    # No hard-coded semantic color is required by the model.
    thickness = (
        3
        if cls_conf >= CLASSIFIER_LOW_CONF
        else 2
    )

    box_color = (
        0,
        255,
        0
    )

    if cls_conf < CLASSIFIER_LOW_CONF:
        box_color = (
            0,
            165,
            255
        )

    cv2.rectangle(
        frame,
        (x1, y1),
        (x2, y2),
        box_color,
        thickness,
    )

    low_tag = (
        ""
        if cls_conf >= CLASSIFIER_LOW_CONF
        else " LOWCONF"
    )

    line1 = (
        f"{species}{low_tag}"
    )

    line2 = (
        f"Det {det_conf:.2f}"
        f" | Cls {cls_conf:.2f}"
    )

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.55
    font_thickness = 2

    (w1, h1), _ = cv2.getTextSize(
        line1,
        font,
        font_scale,
        font_thickness,
    )

    (w2, h2), _ = cv2.getTextSize(
        line2,
        font,
        font_scale,
        font_thickness,
    )

    label_w = max(
        w1,
        w2
    ) + 12

    label_h = (
        h1
        + h2
        + 18
    )

    label_y1 = max(
        0,
        y1 - label_h
    )

    label_y2 = y1

    cv2.rectangle(
        frame,
        (
            x1,
            label_y1,
        ),
        (
            min(
                W - 1,
                x1 + label_w
            ),
            label_y2,
        ),
        box_color,
        -1,
    )

    text_y1 = (
        label_y1
        + h1
        + 4
    )

    text_y2 = (
        text_y1
        + h2
        + 6
    )

    cv2.putText(
        frame,
        line1,
        (
            x1 + 5,
            text_y1,
        ),
        font,
        font_scale,
        (
            0,
            0,
            0
        ),
        font_thickness,
        cv2.LINE_AA,
    )

    cv2.putText(
        frame,
        line2,
        (
            x1 + 5,
            text_y2,
        ),
        font,
        font_scale,
        (
            0,
            0,
            0
        ),
        font_thickness,
        cv2.LINE_AA,
    )


print("STATUS: VIDEO_HELPERS_READY")


In [ ]:
# ============================================================
# CELL 6 — OUTPUT CONFIGURATION
# ============================================================

OUTPUT_DRIVE_ROOT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "Video_Inference_Stage3_MobileNetV4_V3"
)

OUTPUT_DRIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# Work locally first for speed; copy results to Drive after each video.
LOCAL_OUTPUT_ROOT = (
    WORK_ROOT
    / "outputs"
)

LOCAL_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# By default process every frame.
FRAME_STRIDE = 1

# Set to an integer like 1000 for a quick smoke test.
# Keep None for full videos.
MAX_FRAMES_PER_VIDEO = None


print("=" * 110)
print("OUTPUT CONFIG")
print("=" * 110)

print("Drive output:")
print(OUTPUT_DRIVE_ROOT)

print("\nFrame stride:", FRAME_STRIDE)
print("Max frames  :", MAX_FRAMES_PER_VIDEO)

print("\nSTATUS: OUTPUT_CONFIG_READY")


In [ ]:
# ============================================================
# CELL 7 — PROCESS ALL VIDEOS
#
# STREAMING:
# one frame in RAM at a time.
# ============================================================

all_video_summaries = []

print("=" * 120)
print("START VIDEO INFERENCE")
print("=" * 120)


for video_number, video_path in enumerate(
    video_paths,
    start=1
):
    print("\n" + "=" * 120)
    print(
        f"VIDEO {video_number}/{len(video_paths)}"
    )
    print("=" * 120)

    print(
        "Input:",
        video_path.relative_to(INPUT_ROOT)
    )

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        print(
            "WARNING: Could not open video. Skipping:",
            video_path
        )
        continue

    source_fps = float(
        cap.get(
            cv2.CAP_PROP_FPS
        )
    )

    if (
        not np.isfinite(source_fps)
        or source_fps <= 0
    ):
        source_fps = 25.0

    frame_width = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    frame_height = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )

    total_source_frames = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    output_fps = (
        source_fps
        / FRAME_STRIDE
    )

    safe_stem = (
        video_path.stem
        .replace(" ", "_")
    )

    local_video_dir = (
        LOCAL_OUTPUT_ROOT
        / safe_stem
    )

    local_video_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    local_annotated = (
        local_video_dir
        / f"{safe_stem}_annotated.mp4"
    )

    local_csv = (
        local_video_dir
        / f"{safe_stem}_detections.csv"
    )

    fourcc = cv2.VideoWriter_fourcc(
        *"mp4v"
    )

    writer = cv2.VideoWriter(
        str(local_annotated),
        fourcc,
        output_fps,
        (
            frame_width,
            frame_height,
        ),
    )

    if not writer.isOpened():
        cap.release()

        raise RuntimeError(
            f"Could not create output video: {local_annotated}"
        )


    csv_file = open(
        local_csv,
        "w",
        newline="",
        encoding="utf-8",
    )

    csv_writer = csv.DictWriter(
        csv_file,
        fieldnames=[
            "video",
            "source_frame",
            "output_frame",
            "time_sec",
            "object_index",

            "x1",
            "y1",
            "x2",
            "y2",

            "det_conf",

            "species",
            "cls_conf",
            "combined_conf",

            "crop_x1",
            "crop_y1",
            "crop_x2",
            "crop_y2",

            "detector_ms",
            "classifier_ms",
            "frame_pipeline_ms",
        ],
    )

    csv_writer.writeheader()


    source_frame_idx = -1
    output_frame_idx = 0

    total_detections = 0
    low_conf_classifications = 0

    detector_times_ms = []
    classifier_times_ms = []
    pipeline_times_ms = []

    species_counts = defaultdict(
        int
    )

    video_start = time.perf_counter()


    while True:
        ok, frame = cap.read()

        if not ok:
            break

        source_frame_idx += 1

        if (
            source_frame_idx
            % FRAME_STRIDE
            != 0
        ):
            continue

        if (
            MAX_FRAMES_PER_VIDEO is not None
            and output_frame_idx >= MAX_FRAMES_PER_VIDEO
        ):
            break


        frame_start = time.perf_counter()


        # ----------------------------------------------------
        # 1. Detector
        # ----------------------------------------------------

        det_start = time.perf_counter()

        det_result = detector.predict(
            source=frame,
            imgsz=DETECTOR_IMGSZ,
            conf=DETECTOR_CONF,
            iou=DETECTOR_NMS_IOU,
            max_det=300,
            device=YOLO_DEVICE,
            verbose=False,
        )[0]

        detector_ms = (
            time.perf_counter()
            - det_start
        ) * 1000.0


        if (
            det_result.boxes is None
            or len(det_result.boxes) == 0
        ):
            boxes = np.empty(
                (0, 4),
                dtype=np.float32,
            )

            det_confs = np.empty(
                (0,),
                dtype=np.float32,
            )

        else:
            boxes = (
                det_result.boxes.xyxy
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            det_confs = (
                det_result.boxes.conf
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )


        # ----------------------------------------------------
        # 2. Build +10% crops
        # ----------------------------------------------------

        valid_boxes = []
        valid_det_confs = []
        crops = []
        padded_boxes = []


        for box, det_conf in zip(
            boxes,
            det_confs,
        ):
            crop, padded_box = padded_crop_from_xyxy(
                frame,
                box,
                padding=CONTEXT_PAD,
            )

            if crop is None:
                continue

            valid_boxes.append(
                box
            )

            valid_det_confs.append(
                float(det_conf)
            )

            crops.append(
                crop
            )

            padded_boxes.append(
                padded_box
            )


        # ----------------------------------------------------
        # 3. Classifier — detections of this frame as one batch
        # ----------------------------------------------------

        cls_start = time.perf_counter()

        classifications = classify_frame_crops(
            crops
        )

        classifier_ms = (
            time.perf_counter()
            - cls_start
        ) * 1000.0


        # ----------------------------------------------------
        # 4. Draw + CSV
        # ----------------------------------------------------

        for object_idx, (
            box,
            det_conf,
            padded_box,
            cls_output,
        ) in enumerate(
            zip(
                valid_boxes,
                valid_det_confs,
                padded_boxes,
                classifications,
            )
        ):
            species = cls_output[
                "species"
            ]

            cls_conf = cls_output[
                "class_conf"
            ]

            combined_conf = (
                float(det_conf)
                * float(cls_conf)
            )

            if (
                cls_conf
                < CLASSIFIER_LOW_CONF
            ):
                low_conf_classifications += 1

            species_counts[
                species
            ] += 1

            total_detections += 1


            draw_detection_label(
                frame,
                box,
                species,
                float(det_conf),
                float(cls_conf),
            )


            x1, y1, x2, y2 = map(
                float,
                box
            )

            (
                cx1,
                cy1,
                cx2,
                cy2,
            ) = padded_box


            # Filled after total frame timing below using current
            # partial values; frame_pipeline_ms is overwritten in
            # a small temporary record list.
            csv_writer.writerow({
                "video": video_path.name,

                "source_frame": source_frame_idx,
                "output_frame": output_frame_idx,

                "time_sec": (
                    source_frame_idx
                    / source_fps
                ),

                "object_index": object_idx,

                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,

                "det_conf": float(
                    det_conf
                ),

                "species": species,

                "cls_conf": float(
                    cls_conf
                ),

                "combined_conf": combined_conf,

                "crop_x1": cx1,
                "crop_y1": cy1,
                "crop_x2": cx2,
                "crop_y2": cy2,

                "detector_ms": detector_ms,
                "classifier_ms": classifier_ms,

                # Approximate at object-row write time.
                "frame_pipeline_ms": np.nan,
            })


        writer.write(
            frame
        )

        pipeline_ms = (
            time.perf_counter()
            - frame_start
        ) * 1000.0

        detector_times_ms.append(
            detector_ms
        )

        classifier_times_ms.append(
            classifier_ms
        )

        pipeline_times_ms.append(
            pipeline_ms
        )

        output_frame_idx += 1


        if (
            output_frame_idx
            % 100
            == 0
        ):
            mean_ms = float(
                np.mean(
                    pipeline_times_ms[-100:]
                )
            )

            approx_fps = (
                1000.0 / mean_ms
                if mean_ms > 0
                else 0.0
            )

            print(
                f"Processed {output_frame_idx} frames"
                f" | detections={total_detections}"
                f" | recent pipeline={mean_ms:.1f} ms/frame"
                f" | ~{approx_fps:.2f} FPS"
            )


        # Explicitly release current-frame result.
        del det_result

        if output_frame_idx % 50 == 0:
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


    # --------------------------------------------------------
    # Close video resources
    # --------------------------------------------------------

    cap.release()
    writer.release()
    csv_file.close()

    video_wall_sec = (
        time.perf_counter()
        - video_start
    )


    mean_detector_ms = (
        float(
            np.mean(
                detector_times_ms
            )
        )
        if detector_times_ms
        else 0.0
    )

    mean_classifier_ms = (
        float(
            np.mean(
                classifier_times_ms
            )
        )
        if classifier_times_ms
        else 0.0
    )

    mean_pipeline_ms = (
        float(
            np.mean(
                pipeline_times_ms
            )
        )
        if pipeline_times_ms
        else 0.0
    )

    pipeline_fps = (
        1000.0
        / mean_pipeline_ms
        if mean_pipeline_ms > 0
        else 0.0
    )

    avg_dets_per_frame = (
        total_detections
        / output_frame_idx
        if output_frame_idx
        else 0.0
    )


    summary = {
        "video": video_path.name,

        "source_width": frame_width,
        "source_height": frame_height,

        "source_fps": source_fps,
        "source_frames_reported": total_source_frames,

        "processed_frames": output_frame_idx,

        "total_detections": total_detections,
        "avg_detections_per_frame": avg_dets_per_frame,

        "low_conf_classifications": low_conf_classifications,

        "mean_detector_ms": mean_detector_ms,
        "mean_classifier_ms": mean_classifier_ms,
        "mean_pipeline_ms": mean_pipeline_ms,
        "pipeline_fps": pipeline_fps,

        "video_wall_sec": video_wall_sec,

        "annotated_video": str(
            local_annotated
        ),

        "detections_csv": str(
            local_csv
        ),
    }


    for species in idx_to_class.values():
        summary[
            f"count_{species}"
        ] = int(
            species_counts[
                species
            ]
        )


    all_video_summaries.append(
        summary
    )


    # --------------------------------------------------------
    # Copy this video's outputs to Drive immediately.
    # --------------------------------------------------------

    drive_video_dir = (
        OUTPUT_DRIVE_ROOT
        / safe_stem
    )

    if drive_video_dir.exists():
        shutil.rmtree(
            drive_video_dir
        )

    shutil.copytree(
        local_video_dir,
        drive_video_dir,
    )


    summary_json_path = (
        drive_video_dir
        / f"{safe_stem}_summary.json"
    )

    with open(
        summary_json_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            summary,
            f,
            indent=2,
            ensure_ascii=False,
        )


    print("\nCompleted:", video_path.name)
    print("Frames processed :", output_frame_idx)
    print("Detections       :", total_detections)
    print(
        "Mean detector    :",
        f"{mean_detector_ms:.2f} ms/frame"
    )
    print(
        "Mean classifier  :",
        f"{mean_classifier_ms:.2f} ms/frame"
    )
    print(
        "Mean pipeline    :",
        f"{mean_pipeline_ms:.2f} ms/frame"
    )
    print(
        "Approx pipeline  :",
        f"{pipeline_fps:.2f} FPS"
    )

    print("\nSpecies prediction counts:")
    for species in idx_to_class.values():
        print(
            f"  {species:<20}: "
            f"{species_counts[species]}"
        )

    print("\nSaved to Drive:")
    print(drive_video_dir)


print("\nSTATUS: ALL_VIDEOS_PROCESSED")


In [ ]:
# ============================================================
# CELL 8 — FINAL SUMMARY + ZIP ALL OUTPUTS
# ============================================================

summary_df = pd.DataFrame(
    all_video_summaries
)

SUMMARY_CSV = (
    OUTPUT_DRIVE_ROOT
    / "all_video_inference_summary.csv"
)

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
)


GLOBAL_JSON = (
    OUTPUT_DRIVE_ROOT
    / "video_inference_experiment_summary.json"
)

global_summary = {
    "experiment": (
        "Stage3_YOLO11n_MobileNetV4_V3_Video_Inference_V1"
    ),

    "detector_checkpoint": str(
        DETECTOR_CKPT
    ),

    "classifier_checkpoint": str(
        CLASSIFIER_CKPT
    ),

    "detector_imgsz": DETECTOR_IMGSZ,
    "detector_conf": DETECTOR_CONF,
    "detector_nms_iou": DETECTOR_NMS_IOU,

    "classifier_input": CLASSIFIER_INPUT,
    "context_padding": CONTEXT_PAD,

    "classifier_low_conf_mark": CLASSIFIER_LOW_CONF,

    "input_zip": str(
        VIDEO_ZIP
    ),

    "videos_processed": int(
        len(summary_df)
    ),

    "total_frames_processed": int(
        summary_df["processed_frames"].sum()
        if len(summary_df)
        else 0
    ),

    "total_detections": int(
        summary_df["total_detections"].sum()
        if len(summary_df)
        else 0
    ),

    "mean_pipeline_fps_unweighted": float(
        summary_df["pipeline_fps"].mean()
        if len(summary_df)
        else 0.0
    ),

    "output_root": str(
        OUTPUT_DRIVE_ROOT
    ),

    "training_performed": False,
}

with open(
    GLOBAL_JSON,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        global_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# Create one ZIP with all generated outputs.
# ------------------------------------------------------------

ZIP_BASE = Path(
    "/content/Coral_Video_Inference_Stage3_MobileNetV4_V3"
)

zip_path = Path(
    shutil.make_archive(
        str(ZIP_BASE),
        "zip",
        root_dir=str(OUTPUT_DRIVE_ROOT),
    )
)

DRIVE_ZIP = (
    OUTPUT_DRIVE_ROOT
    / zip_path.name
)

shutil.copy2(
    zip_path,
    DRIVE_ZIP,
)


print("=" * 120)
print("VIDEO INFERENCE COMPLETE")
print("=" * 120)

if len(summary_df):
    display_cols = [
        "video",
        "processed_frames",
        "total_detections",
        "avg_detections_per_frame",
        "mean_detector_ms",
        "mean_classifier_ms",
        "mean_pipeline_ms",
        "pipeline_fps",
    ]

    print(
        summary_df[
            display_cols
        ].to_string(
            index=False,
        )
    )

print("\nOutput folder:")
print(OUTPUT_DRIVE_ROOT)

print("\nGlobal summary CSV:")
print(SUMMARY_CSV)

print("\nGlobal summary JSON:")
print(GLOBAL_JSON)

print("\nZIP of all outputs:")
print(DRIVE_ZIP)

print("\nSTATUS: VIDEO_INFERENCE_COMPLETE")


In [ ]:
# ============================================================
# CELL 9 — OPTIONAL: PREVIEW FIRST ANNOTATED VIDEO IN COLAB
# ============================================================

from IPython.display import Video, display

annotated_candidates = sorted(
    OUTPUT_DRIVE_ROOT.rglob(
        "*_annotated.mp4"
    )
)

if not annotated_candidates:
    print(
        "No annotated videos found yet."
    )

else:
    preview_video = annotated_candidates[0]

    print(
        "Preview:",
        preview_video
    )

    display(
        Video(
            str(preview_video),
            embed=True,
            width=900,
        )
    )


## Later detector replacement

When Stage 3.1 or Stage 4.1 finishes, keep this whole notebook unchanged and replace only `DETECTOR_CKPT` in Cell 2.

That gives a clean video comparison because:

- same video
- same MobileNetV4 V3 classifier
- same `+10%` crop context
- same 288 input
- same drawing logic
- same timing code

Only the detector changes.
